# Notebook 02: DLS Method Implementation & Validation

Implements the parametric DLS model, fits parameters from actual match data, and validates predictions.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

from src.data_collection import load_processed
from src.data_processing import filter_completed_matches, create_over_snapshots, add_match_context
from src.dls_method import DLSModel
from src.feature_engineering import add_dls_features
from src.evaluation import compute_metrics
from src.visualizations import plot_dls_resource_curves

## 2.1 Load and Prepare Data

In [2]:
matches_df, deliveries_df = load_processed('mens_odi')
matches_df, deliveries_df = filter_completed_matches(matches_df, deliveries_df)

print(f"Completed matches: {len(matches_df)}")
print(f"Deliveries: {len(deliveries_df)}")

2026-02-05 22:49:52,413 - INFO - Filtered from 3085 to 2709 completed matches (removed 376 D/L, no-result, or incomplete)


Completed matches: 2709
Deliveries: 1486063


In [3]:
# Create over-boundary snapshots
snapshots = create_over_snapshots(deliveries_df, matches_df, overs_limit=50)
snapshots = add_match_context(snapshots, matches_df)
print(f"Snapshots: {len(snapshots)} rows from {snapshots['match_id'].nunique()} matches")
snapshots.head()

2026-02-05 22:54:49,871 - INFO - Created 129322 over-boundary snapshots from 2709 matches


Snapshots: 129322 rows from 2709 matches


,match_id,batting_team,overs_completed,overs_remaining,current_score,wickets_fallen,wickets_in_hand,current_run_rate,recent_run_rate_5,scoring_acceleration,...,cumulative_sixes,innings_progress,final_total,date,venue,city,toss_winner,toss_decision,year,toss_bat_first
0,1000887,Australia,1,49,1,0,10,1.0000,1.0000,0.0,...,0,0.02,268,2017-01-13,"Brisbane Cricket Ground, Woolloongabba",Brisbane,Australia,bat,2017,1
1,1000887,Australia,2,48,2,0,10,1.0000,1.0000,0.0,...,0,0.04,268,2017-01-13,"Brisbane Cricket Ground, Woolloongabba",Brisbane,Australia,bat,2017,1
2,1000887,Australia,3,47,5,0,10,1.6667,1.6667,0.0,...,0,0.06,268,2017-01-13,"Brisbane Cricket Ground, Woolloongabba",Brisbane,Australia,bat,2017,1
3,1000887,Australia,4,46,11,0,10,2.7500,2.7500,0.0,...,0,0.08,268,2017-01-13,"Brisbane Cricket Ground, Woolloongabba",Brisbane,Australia,bat,2017,1
4,1000887,Australia,5,45,13,2,8,2.6000,2.6000,0.0,...,0,0.10,268,2017-01-13,"Brisbane Cricket Ground, Woolloongabba",Brisbane,Australia,bat,2017,1


## 2.2 Fit DLS Model

In [4]:
dls = DLSModel(overs_limit=50)
dls.fit(snapshots)

print(f"\nFitted G50 (average uninterrupted score): {dls.G50:.1f}")
print(f"\nFitted Z0 parameters (max runs by wickets lost):")
for w in range(10):
    print(f"  {w} wickets: Z0={dls.Z0[w]:.1f}, b={dls.b[w]:.4f}")

2026-02-05 22:55:07,648 - INFO - Wickets=0: Z0=285.79, b=0.0405 (fitted from 18344 points)
2026-02-05 22:55:07,674 - INFO - Wickets=1: Z0=242.77, b=0.0485 (fitted from 20648 points)
2026-02-05 22:55:07,702 - INFO - Wickets=2: Z0=210.63, b=0.0549 (fitted from 21303 points)
2026-02-05 22:55:07,733 - INFO - Wickets=3: Z0=170.42, b=0.0677 (fitted from 18644 points)
2026-02-05 22:55:07,755 - INFO - Wickets=4: Z0=136.56, b=0.0835 (fitted from 15689 points)
2026-02-05 22:55:07,772 - INFO - Wickets=5: Z0=107.85, b=0.1008 (fitted from 11998 points)
2026-02-05 22:55:07,791 - INFO - Wickets=6: Z0=79.10, b=0.1359 (fitted from 8360 points)
2026-02-05 22:55:07,805 - INFO - Wickets=7: Z0=51.42, b=0.2056 (fitted from 5720 points)
2026-02-05 22:55:07,817 - INFO - Wickets=8: Z0=31.08, b=0.2831 (fitted from 3768 points)
2026-02-05 22:55:07,827 - INFO - Wickets=9: Z0=12.63, b=0.5941 (fitted from 2198 points)
2026-02-05 22:55:07,828 - INFO - Fitted G50 = 248.1



Fitted G50 (average uninterrupted score): 248.1

Fitted Z0 parameters (max runs by wickets lost):
  0 wickets: Z0=285.8, b=0.0405
  1 wickets: Z0=242.8, b=0.0485
  2 wickets: Z0=210.6, b=0.0549
  3 wickets: Z0=170.4, b=0.0677
  4 wickets: Z0=136.6, b=0.0835
  5 wickets: Z0=107.9, b=0.1008
  6 wickets: Z0=79.1, b=0.1359
  7 wickets: Z0=51.4, b=0.2056
  8 wickets: Z0=31.1, b=0.2831
  9 wickets: Z0=12.6, b=0.5941


In [5]:
# Save the fitted model
dls.save()
print("DLS model saved.")

2026-02-05 22:55:08,855 - INFO - DLS model saved to /Users/akshma/Downloads/rajat_thesis/results/models/dls_model_50.pkl


DLS model saved.


## 2.3 DLS Resource Table

In [6]:
# Generate standard DLS resource table
resource_table = dls.get_resource_table()
print("DLS Resource Table (% remaining):")
# Show every 5 overs
display_overs = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
resource_table.loc[display_overs]

DLS Resource Table (% remaining):


,w=0,w=1,w=2,w=3,w=4,w=5,w=6,w=7,w=8,w=9
overs_remaining,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,21.1,21.1,20.4,19.7,18.8,17.2,15.7,13.3,9.5,4.8
10,38.4,37.6,35.9,33.8,31.2,27.6,23.7,18.1,11.8,5.1
15,52.5,50.6,47.6,43.8,39.3,33.9,27.7,19.8,12.3,5.1
20,64.0,60.8,56.6,51.0,44.7,37.7,29.8,20.4,12.5,5.1
25,73.4,68.7,63.4,56.0,48.2,40.0,30.8,20.6,12.5,5.1
30,81.0,75.0,68.5,59.7,50.6,41.4,31.3,20.7,12.5,5.1
35,87.3,79.9,72.5,62.3,52.1,42.2,31.6,20.7,12.5,5.1
40,92.4,83.8,75.5,64.1,53.1,42.7,31.7,20.7,12.5,5.1


## 2.4 DLS Resource Curves Visualization

In [7]:
# Plot resource curves
plot_dls_resource_curves(dls, format_key='mens_odi')

# Also display inline
fig, ax = plt.subplots(figsize=(12, 7))
overs_range = np.arange(0, 51, 0.5)

cmap = plt.cm.viridis(np.linspace(0, 0.9, 10))
for w in range(10):
    resources = [dls.resource_remaining(u, w) for u in overs_range]
    ax.plot(overs_range, resources, label=f'{w} wkts', color=cmap[w], linewidth=2)

ax.set_xlabel('Overs Remaining', fontsize=13)
ax.set_ylabel('Resources Remaining (%)', fontsize=13)
ax.set_title('Fitted DLS Resource Curves - Men\'s ODI', fontsize=15)
ax.legend(title='Wickets Lost', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

2026-02-05 22:55:12,142 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_dls_resource_curves.png


## 2.5 DLS Predictions vs Actual

In [8]:
# Add DLS predictions to snapshots
snapshots_dls = add_dls_features(snapshots, dls)

# Evaluate DLS predictions overall
valid = snapshots_dls.dropna(subset=['dls_predicted_final'])
valid = valid[valid['overs_completed'] >= 5]  # Need some data for reasonable predictions

dls_metrics = compute_metrics(valid['final_total'].values, valid['dls_predicted_final'].values)
print("DLS Prediction Metrics (overs >= 5):")
for k, v in dls_metrics.items():
    print(f"  {k}: {v}")

DLS Prediction Metrics (overs >= 5):
  RMSE: 49.83
  MAE: 34.32
  R2: 0.4151
  MAPE: 14.79
  Within_5: 15.78
  Within_10: 28.4
  Within_20: 46.87
  N: 118486


In [9]:
# DLS accuracy by match phase
phases = {
    'Early (1-10)': (1, 10),
    'Middle (11-30)': (11, 30),
    'Late (31-40)': (31, 40),
    'Death (41-50)': (41, 50),
}

print("\nDLS Performance by Phase:")
print(f"{'Phase':<20} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'Within 10':>10}")
print('-' * 60)

for phase_name, (lo, hi) in phases.items():
    mask = (valid['overs_completed'] >= lo) & (valid['overs_completed'] <= hi)
    subset = valid[mask]
    if len(subset) > 0:
        m = compute_metrics(subset['final_total'].values, subset['dls_predicted_final'].values)
        print(f"{phase_name:<20} {m['RMSE']:>8.1f} {m['MAE']:>8.1f} {m['R2']:>8.3f} {m['Within_10']:>9.1f}%")


DLS Performance by Phase:
Phase                    RMSE      MAE       R²  Within 10
------------------------------------------------------------
Early (1-10)             84.9     66.7   -0.531      11.0%
Middle (11-30)           53.1     40.9    0.384      17.2%
Late (31-40)             28.4     21.8    0.795      31.2%
Death (41-50)            13.8      9.7    0.941      64.0%


In [10]:
# Actual vs DLS Predicted scatter
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full scatter
ax = axes[0]
ax.scatter(valid['final_total'], valid['dls_predicted_final'], alpha=0.1, s=5, color='steelblue')
ax.plot([0, 500], [0, 500], 'r--', linewidth=2, label='Perfect')
ax.set_xlabel('Actual First Innings Total')
ax.set_ylabel('DLS Predicted Total')
ax.set_title('DLS: Actual vs Predicted')
ax.legend()
ax.grid(True, alpha=0.3)

# Error by over
ax = axes[1]
errors = valid.copy()
errors['abs_error'] = (errors['final_total'] - errors['dls_predicted_final']).abs()
over_errors = errors.groupby('overs_completed')['abs_error'].mean()
ax.plot(over_errors.index, over_errors.values, 'b-o', markersize=4, linewidth=2)
ax.set_xlabel('Overs Completed')
ax.set_ylabel('Mean Absolute Error')
ax.set_title('DLS Prediction Error by Over')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [11]:
# Save snapshots with DLS features for use in ML models
from src.data_processing import PROCESSED_DIR, time_based_train_test_split

snapshots_dls.to_parquet(PROCESSED_DIR / 'mens_odi_snapshots.parquet', index=False)
train_df, test_df = time_based_train_test_split(snapshots_dls, matches_df)
train_df.to_parquet(PROCESSED_DIR / 'mens_odi_train.parquet', index=False)
test_df.to_parquet(PROCESSED_DIR / 'mens_odi_test.parquet', index=False)

print(f"Saved train ({len(train_df)} rows) and test ({len(test_df)} rows) with DLS features.")
print("\n✓ DLS analysis complete!")

2026-02-05 22:55:26,563 - INFO - Train/Test split: 2167 train matches (103930 rows), 542 test matches (25392 rows). Cutoff date: 2022-12-06


Saved train (103930 rows) and test (25392 rows) with DLS features.

✓ DLS analysis complete!
